# Preprocesamiento de Texto - Reviews de Hoteles en Europa

Dataset: [515K Hotel Reviews Data in Europe](https://www.kaggle.com/datasets/jiashenliu/515k-hotel-reviews-data-in-europe)

Columnas: `Reviewer_Nationality`, `Negative_Review`, `Positive_Review`

In [1]:
import kagglehub
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer

nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

path = kagglehub.dataset_download('jiashenliu/515k-hotel-reviews-data-in-europe')
df = pd.read_csv(f'{path}/Hotel_Reviews.csv')

print(f'Shape: {df.shape}')
print(f'Columnas: {len(df.columns)}')
print()
df[['Reviewer_Nationality', 'Negative_Review', 'Positive_Review']].head()

/home/jheisonmblivecom/Academic/Deriverables/ML/nlp-hotel-reviews/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Shape: (515738, 17)
Columnas: 17



,Reviewer_Nationality,Negative_Review,Positive_Review
0,Russia,I am so angry that i made this post available...,Only the park outside of the hotel was beauti...
1,Ireland,No Negative,No real complaints the hotel was great great ...
2,Australia,Rooms are nice but for elderly a bit difficul...,Location was good and staff were ok It is cut...
3,United Kingdom,My room was dirty and I was afraid to walk ba...,Great location in nice surroundings the bar a...
4,New Zealand,You When I booked with your company on line y...,Amazing location and building Romantic setting


## 1. Carga del dataset

### Estructura y nulos

In [2]:
df[['Reviewer_Nationality', 'Negative_Review', 'Positive_Review']].info()

<class 'pandas.DataFrame'>
RangeIndex: 515738 entries, 0 to 515737
Data columns (total 3 columns):
 #   Column                Non-Null Count   Dtype
---  ------                --------------   -----
 0   Reviewer_Nationality  515738 non-null  str  
 1   Negative_Review       515738 non-null  str  
 2   Positive_Review       515738 non-null  str  
dtypes: str(3)
memory usage: 11.8 MB


In [3]:
df[['Reviewer_Nationality', 'Negative_Review', 'Positive_Review']].isnull().sum()

Reviewer_Nationality    0
Negative_Review         0
Positive_Review         0
dtype: int64

### Top 10 nacionalidades

In [4]:
df['Reviewer_Nationality'].value_counts().head(10)

Reviewer_Nationality
United Kingdom               245246
United States of America      35437
Australia                     21686
Ireland                       14827
United Arab Emirates          10235
Saudi Arabia                   8951
Netherlands                    8772
Switzerland                    8678
Germany                        7941
Canada                         7894
Name: count, dtype: int64

### Ejemplos de reseñas

In [5]:
for i, row in df[['Negative_Review', 'Positive_Review']].head(3).iterrows():
    print(f'--- Review {i} ---')
    print(f'Negativa: {row["Negative_Review"][:200]}')
    print(f'Positiva: {row["Positive_Review"][:200]}')
    print()

--- Review 0 ---
Negativa:  I am so angry that i made this post available via all possible sites i use when planing my trips so no one will make the mistake of booking this place I made my booking via booking com We stayed for 
Positiva:  Only the park outside of the hotel was beautiful 

--- Review 1 ---
Negativa: No Negative
Positiva:  No real complaints the hotel was great great location surroundings rooms amenities and service Two recommendations however firstly the staff upon check in are very confusing regarding deposit payment

--- Review 2 ---
Negativa:  Rooms are nice but for elderly a bit difficult as most rooms are two story with narrow steps So ask for single level Inside the rooms are very very basic just tea coffee and boiler and no bar empty f
Positiva:  Location was good and staff were ok It is cute hotel the breakfast range is nice Will go back 



## 2. Limpieza con regex

Observamos el efecto de la limpieza: minúsculas, sin caracteres especiales, espacios normalizados.

In [6]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

sample = df['Positive_Review'].iloc[0]
print('Original:', sample[:150])
print()
print('Limpio:  ', clean_text(sample)[:150])

Original:  Only the park outside of the hotel was beautiful 

Limpio:   only the park outside of the hotel was beautiful


## 3. Tokenización

Separamos el texto limpio en tokens individuales.

In [7]:
def tokenize_text(text):
    return word_tokenize(text)

sample_clean = clean_text(df['Positive_Review'].iloc[0])
tokens = tokenize_text(sample_clean)
print('Tokens:', tokens[:20])
print(f'Total tokens: {len(tokens)}')

Tokens: ['only', 'the', 'park', 'outside', 'of', 'the', 'hotel', 'was', 'beautiful']
Total tokens: 9


## 4. Stopwords

Eliminamos palabras de parada en inglés (the, is, was, at, etc.).

In [8]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [t for t in tokens if t not in stop_words]

tokens_no_stop = remove_stopwords(tokens)
print('Antes:', tokens[:15])
print()
print('Después:', tokens_no_stop[:15])
print(f'Eliminadas: {len(tokens) - len(tokens_no_stop)} / {len(tokens)}')

Antes: ['only', 'the', 'park', 'outside', 'of', 'the', 'hotel', 'was', 'beautiful']

Después: ['park', 'outside', 'hotel', 'beautiful']
Eliminadas: 5 / 9


## 5. Bolsa de palabras - reseñas positivas

Top palabras más frecuentes en reseñas positivas después del preprocesamiento completo.

In [9]:
df_temp = df.head(5000).copy()
df_temp['Positive_Review'] = df_temp['Positive_Review'].replace('No Positive', '')

bow_sample = df_temp['Positive_Review'].apply(lambda x: ' '.join(remove_stopwords(tokenize_text(clean_text(x)))))

vectorizer = CountVectorizer(max_features=500)
bow_matrix = vectorizer.fit_transform(bow_sample)

print(f'Matriz BoW: {bow_matrix.shape}')
print(f'Top 20 palabras más frecuentes (reseñas positivas):')
freqs = bow_matrix.sum(axis=0).A1
words = vectorizer.get_feature_names_out()
top_idx = freqs.argsort()[-20:][::-1]
for idx in top_idx:
    print(f'  {words[idx]}: {int(freqs[idx])}')

Matriz BoW: (5000, 500)
Top 20 palabras más frecuentes (reseñas positivas):
  staff: 2046
  location: 1789
  room: 1599
  hotel: 1263
  good: 1105
  great: 960
  friendly: 914
  helpful: 797
  comfortable: 703
  breakfast: 692
  nice: 663
  clean: 623
  excellent: 565
  bed: 559
  lovely: 490
  rooms: 390
  quiet: 371
  stay: 355
  london: 323
  station: 323


## 6. Pipeline completo

Aplicamos limpieza + tokenización + stopwords a ambas columnas.

In [10]:
df_sample = df.head(10000).copy()

df_sample['Negative_Review'] = df_sample['Negative_Review'].replace('No Negative', '')
df_sample['Positive_Review'] = df_sample['Positive_Review'].replace('No Positive', '')

def full_pipeline(text):
    text = clean_text(text)
    tokens = tokenize_text(text)
    tokens = remove_stopwords(tokens)
    return ' '.join(tokens)

df_sample['Clean_Positive'] = df_sample['Positive_Review'].apply(full_pipeline)
df_sample['Clean_Negative'] = df_sample['Negative_Review'].apply(full_pipeline)

print('Ejemplo pipeline completo:')
print(f'Original:  {df_sample["Positive_Review"].iloc[0][:120]}')
print(f'Procesado: {df_sample["Clean_Positive"].iloc[0][:120]}')

Ejemplo pipeline completo:
Original:   Only the park outside of the hotel was beautiful 
Procesado: park outside hotel beautiful


## 7. Bolsa de palabras - reseñas negativas

Top palabras más frecuentes en reseñas negativas. Comparamos con la bolsa positiva para identificar qué molesta a los huéspedes.

In [11]:
neg_sample = df_sample['Clean_Negative'][df_sample['Clean_Negative'].str.len() > 0]

vectorizer_neg = CountVectorizer(max_features=500)
bow_neg = vectorizer_neg.fit_transform(neg_sample)

print(f'Matriz BoW negativa: {bow_neg.shape}')
print(f'Top 20 palabras en reseñas negativas:')
freqs = bow_neg.sum(axis=0).A1
words = vectorizer_neg.get_feature_names_out()
top_idx = freqs.argsort()[-20:][::-1]
for idx in top_idx:
    print(f'  {words[idx]}: {int(freqs[idx])}')

Matriz BoW negativa: (7806, 500)
Top 20 palabras en reseñas negativas:
  room: 4891
  small: 1758
  hotel: 1629
  breakfast: 1188
  bed: 883
  bathroom: 861
  rooms: 849
  staff: 812
  could: 736
  would: 730
  shower: 642
  one: 638
  nothing: 637
  night: 569
  us: 561
  bit: 506
  little: 502
  like: 493
  get: 459
  even: 427


## 8. Conclusiones

- Dataset: 515,738 reseñas, 17 columnas, sin nulos en las columnas de interés.
- Los valores `"No Negative"` y `"No Positive"` se filtraron antes del preprocesamiento.
- **Reseñas positivas**: dominan `staff`, `location`, `room`, `hotel`, `good`, `great` → los huéspedes valoran el servicio y la ubicación.
- **Reseñas negativas**: dominan `room`, `hotel`, `staff`, `small`, `bed`, `bathroom`, `noise` → las quejas se centran en el tamaño de las habitaciones, el baño y el ruido.
- La bolsa de palabras por separado (positiva vs negativa) permite identificar qué tema genera satisfacción vs molestia, a diferencia de una bolsa mixta que solo muestra frecuencia general.
- Limitación: BoW pierde contexto ('no good' se cuenta como positivo). No se aplicó lemmatización.